#### Imports and Load Data

In [ ]:
import os
import sys
import libpysal
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from esda.moran import Moran
from scipy.stats import spearmanr, chi2_contingency

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
TRACTS_GI_PATH = os.path.join(config.DATA_DIR, "tracts_plot_gi.gpkg")
if not os.path.exists(TRACTS_GI_PATH):
    raise FileNotFoundError("Run 03_spatial_analysis.ipynb first (creates data/tracts_plot_gi.gpkg).")
tracts_plot = gpd.read_file(TRACTS_GI_PATH).to_crs(config.SPATIAL_CRS)
tracts_plot["GEOID"] = tracts_plot["GEOID"].astype(str).str.zfill(11)
tracts_plot["is_hotspot"] = tracts_plot["is_hotspot"].astype(bool)
tracts_plot["is_hotspot_p05"] = tracts_plot["is_hotspot_p05"].astype(bool)
tracts_plot["land_km2"] = tracts_plot["ALAND"] / 1e6
# Two hotspot definitions: FDR-controlled (primary) and uncorrected HH p <= 0.05 (sensitivity, larger set)
HOTSPOT_DEFS = {"HH FDR q<=0.05": "is_hotspot", "HH p<=0.05 (uncorrected)": "is_hotspot_p05"}
for lab, col in HOTSPOT_DEFS.items():
    print(f"Hotspots, {lab}: {tracts_plot[col].sum()} of {len(tracts_plot):,} city tracts")

osm = gpd.read_file(config.OSM_POI).to_crs(config.SPATIAL_CRS)
print(f"OSM POIs (county): {len(osm):,}")

#### Classify OSM POIs

Transit includes railway stations/stops, bus stops and `public_transport` stops/platforms/stations;
`building=retail/commercial` counts as retail / commercial zone.

In [ ]:
def _v(row, key):
    v = row.get(key, None)
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v).lower().strip()


def classify_poi(row):
    amenity, railway, highway = _v(row, "amenity"), _v(row, "railway"), _v(row, "highway")
    parking, shop, landuse = _v(row, "parking"), _v(row, "shop"), _v(row, "landuse")
    building, pt = _v(row, "building"), _v(row, "public_transport")
    if (railway in ("station", "halt", "tram_stop", "subway_entrance") or highway == "bus_stop"
            or amenity in ("bus_station", "ferry_terminal") or pt in ("stop_position", "platform", "station")):
        return "transit"
    if amenity in ("parking", "parking_space", "parking_entrance", "motorcycle_parking") or parking not in ("", "no"):
        return "parking"
    if amenity in ("fast_food", "restaurant", "bar", "cafe", "nightclub", "pub", "food_court"):
        return "food_drink"
    if amenity in ("bank", "atm"):
        return "financial"
    if shop not in ("", "no") or landuse == "retail" or building == "retail":
        return "retail"
    if landuse == "commercial" or building == "commercial":
        return "commercial_zone"
    return "other"


osm["poi_type"] = osm.apply(classify_poi, axis=1)
poi_types = ["transit", "parking", "food_drink", "financial", "retail", "commercial_zone"]
print(osm["poi_type"].value_counts().to_string())

In [ ]:
poi_joined = gpd.sjoin(osm[["poi_type", "geometry"]], tracts_plot[["GEOID", "geometry"]], how="inner", predicate="within")
poi_counts = (poi_joined.groupby(["GEOID", "poi_type"]).size().unstack(fill_value=0)
              .reindex(index=tracts_plot["GEOID"], columns=poi_types, fill_value=0))
tracts_poi = tracts_plot[["GEOID", "is_hotspot", "is_hotspot_p05", "land_km2", "geometry"]].merge(
    (poi_counts > 0).astype(int).reset_index(), on="GEOID", how="left")
print("POI presence (% of city tracts):")
print((tracts_poi[poi_types].mean() * 100).round(1).to_string())

#### Hotspot status vs POI presence

Presence of any POI in a tract depends strongly on tract size, so each POI type is tested twice:
the crude 2x2 chi-square / odds ratio, and a logistic regression of hotspot status on presence controlling for
log land area. p-values are Benjamini-Hochberg adjusted across the six POI types. Tracts are spatially
dependent (hotspots are defined by their neighbours), so these p-values remain optimistic.

In [ ]:
X_area = np.log(tracts_poi["land_km2"].clip(lower=0.01))
parts = []
for def_label, hot_col in HOTSPOT_DEFS.items():
    rows = []
    hot = tracts_poi[hot_col].astype(bool)
    for pt in poi_types:
        ct = pd.crosstab(hot, tracts_poi[pt]).reindex(index=[False, True], columns=[0, 1], fill_value=0)
        chi2, p, _, _ = chi2_contingency(ct, correction=True)
        a, b, c, d = ct.loc[True, 1], ct.loc[True, 0], ct.loc[False, 1], ct.loc[False, 0]
        X = sm.add_constant(pd.DataFrame({"presence": tracts_poi[pt], "log_area": X_area}))
        or_adj, ci, p_adj = np.nan, [np.nan, np.nan], np.nan
        if (ct.values > 0).all():  # an empty cell (e.g. every hotspot has the POI) = separation: OR not estimable
            fit = sm.Logit(hot.astype(int), X).fit(disp=0)
            or_adj, ci, p_adj = np.exp(fit.params["presence"]), np.exp(fit.conf_int().loc["presence"]).values, fit.pvalues["presence"]
        rows.append({"hotspot_definition": def_label, "n_hotspots": int(hot.sum()), "POI type": pt, "chi2": chi2,
                     "p_crude": p, "OR_crude": (a * d) / (b * c) if b * c else np.nan,
                     "OR_area_adjusted": or_adj, "OR_adj_CI_low": ci[0], "OR_adj_CI_high": ci[1], "p_adjusted_model": p_adj,
                     "hotspot_with_POI%": a / (a + b) * 100, "other_with_POI%": c / (c + d) * 100})
    t = pd.DataFrame(rows)
    t["q_crude_BH"] = config.bh_fdr(t["p_crude"])
    t["q_adjusted_BH"] = config.bh_fdr(t["p_adjusted_model"].fillna(1.0))
    parts.append(t)
table5 = pd.concat(parts, ignore_index=True)
table5.to_csv(os.path.join(config.TABLES_DIR, "table5_chi2_poi.csv"), index=False)
print(table5.round(4).to_string(index=False))

In [ ]:
t_p05 = table5[table5["hotspot_definition"] == "HH p<=0.05 (uncorrected)"]
x = np.arange(len(poi_types))
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
c_fdr = table5[table5["hotspot_definition"] == "HH FDR q<=0.05"].set_index("POI type").loc[poi_types]
c_p05 = t_p05.set_index("POI type").loc[poi_types]
other = tracts_poi.loc[~tracts_poi["is_hotspot_p05"], poi_types].mean() * 100
ax.bar(x - 0.27, c_fdr["hotspot_with_POI%"], 0.27, color="#b2182b", alpha=0.85, label=f"Hotspots, FDR (n={int(c_fdr['n_hotspots'].iloc[0])})")
ax.bar(x, c_p05["hotspot_with_POI%"], 0.27, color="#ef8a62", alpha=0.85, label=f"Hotspots, p<=0.05 (n={int(c_p05['n_hotspots'].iloc[0])})")
ax.bar(x + 0.27, other.values, 0.27, color="#7f8c8d", alpha=0.85, label="Other city tracts")
for i, pt in enumerate(poi_types):
    or_adj = c_p05.loc[pt, "OR_area_adjusted"]
    ax.text(i, 102, f"adj. OR {or_adj:.2f}" if pd.notna(or_adj) else "adj. OR n/a", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([t.replace("_", "\n").title() for t in poi_types])
ax.set_ylabel("% of tracts with POI present")
ax.set_ylim(0, 110)
ax.set_title("POI Presence: Hotspot vs Other City Tracts (adj. OR: p<=0.05 definition, area-adjusted)", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig17_poi_presence_hotspot.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Theft vs median household income

Income: vintages re-allocated to 2020 tracts and interpolated to each year, in 2022 dollars (CPI-U), averaged over the 23 theft
years. Tracts: city tracts with a vehicle denominator in every year (>= 100 on average), special land-use tracts
(98xx) excluded.

Household vehicles per resident rise steeply with income, so a rate per household vehicle builds part of the
income gradient into its denominator. The headline estimate is therefore a negative-binomial model of the
23-year theft count with exposure entered as covariates (log household vehicles, log population, log land area)
rather than as a fixed denominator. Model 2 adds OpenStreetMap POI densities (a 2026 snapshot; descriptive only).
Spearman correlations are reported for reference. Tracts are spatially dependent (see the residual Moran's I),
so model-based p-values and standard errors are optimistic; the models also report standard errors clustered by
LAPD geographic area (21 areas; each tract assigned to the area that reports most of its thefts).

In [ ]:
inc = config.annual_denominator("income", tracts_plot["GEOID"].tolist())
analysis = tracts_plot[["GEOID", "total_count", "avg_annual", "rate_per_1000", "rate_per_100k", "mean_vehicles",
                        "mean_population", "land_km2", "gi_tier", "geometry"]].copy()
analysis["median_income"] = inc.mean(axis=1, skipna=True).values
complete = analysis.dropna(subset=["rate_per_1000", "median_income", "mean_population"])
complete = complete[(complete["mean_population"] > 0) & (complete["land_km2"] > 0)].copy()
th = config.load_theft_data(coords_required=True)
rd = pd.to_numeric(th["RPT_DIST_NBR"], errors="coerce")
th["lapd_area"] = (rd // 100).where(rd.between(100, 2199))
tract_area = th.dropna(subset=["lapd_area", "GEOID"]).groupby("GEOID")["lapd_area"].agg(lambda s: int(s.mode().iloc[0]))
complete["lapd_area"] = complete["GEOID"].map(tract_area)
complete = complete.dropna(subset=["lapd_area"])
print(f"Tracts in income analysis: {len(complete)} in {complete['lapd_area'].nunique()} LAPD areas")

rows6 = []
for label, col in [("avg annual count", "avg_annual"), ("rate per 1,000 household vehicles", "rate_per_1000"),
                   ("rate per 100,000 residents", "rate_per_100k")]:
    r = spearmanr(complete["median_income"], complete[col], nan_policy="omit")
    rows6.append({"measure": f"Spearman: income vs {label}", "estimate": r.statistic, "se": np.nan, "p_value": r.pvalue,
                  "se_cluster_lapd_area": np.nan, "p_cluster_lapd_area": np.nan})
r = spearmanr(complete["median_income"], complete["mean_vehicles"] / complete["mean_population"])
rows6.append({"measure": "Spearman: income vs household vehicles per resident", "estimate": r.statistic, "se": np.nan,
              "p_value": r.pvalue, "se_cluster_lapd_area": np.nan, "p_cluster_lapd_area": np.nan})

X1 = pd.DataFrame({"log_vehicles": np.log(complete["mean_vehicles"]), "log_population": np.log(complete["mean_population"]),
                   "log_land_km2": np.log(complete["land_km2"]), "log_income": np.log(complete["median_income"])},
                  index=complete.index)
poi_density = np.log1p(poi_counts.reindex(complete["GEOID"]).values / complete["land_km2"].values[:, None])
X2 = X1.join(pd.DataFrame(poi_density, columns=[f"log1p_{c}_per_km2" for c in poi_types], index=complete.index))
models = {}
for name, X in [("NB model 1 (exposure + income)", X1), ("NB model 2 (+ POI densities)", X2)]:
    model = sm.NegativeBinomial(complete["total_count"].values, sm.add_constant(X))
    fit = model.fit(disp=0, maxiter=200)
    fit_cl = model.fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": complete["lapd_area"].astype(int).values})
    models[name] = fit
    for var in X.columns:
        rows6.append({"measure": f"{name}: {var}", "estimate": fit.params[var], "se": fit.bse[var], "p_value": fit.pvalues[var],
                      "se_cluster_lapd_area": fit_cl.bse[var], "p_cluster_lapd_area": fit_cl.pvalues[var]})
    rows6.append({"measure": f"{name}: dispersion alpha", "estimate": fit.params["alpha"], "se": fit.bse["alpha"], "p_value": np.nan,
                  "se_cluster_lapd_area": np.nan, "p_cluster_lapd_area": np.nan})

nb1 = models["NB model 1 (exposure + income)"]
mu = nb1.predict()
complete["residual_log"] = np.log((complete["total_count"] + 0.5) / (mu + 0.5))
w_res = libpysal.weights.Queen.from_dataframe(complete, use_index=False)
w_res.transform = "r"
np.random.seed(42)
mi_res = Moran(complete["residual_log"].values, w_res, permutations=999)
rows6.append({"measure": "Moran's I of NB model 1 log residuals", "estimate": mi_res.I, "se": np.nan, "p_value": mi_res.p_sim,
              "se_cluster_lapd_area": np.nan, "p_cluster_lapd_area": np.nan})
table6 = pd.DataFrame(rows6)
table6["n_tracts"] = len(complete)
table6.to_csv(os.path.join(config.TABLES_DIR, "table6_income_models.csv"), index=False)
print(table6.round(4).to_string(index=False))
b_inc = nb1.params["log_income"]
print(f"Income elasticity (NB model 1): {b_inc:.2f}; a 10 % higher median income goes with "
      f"{(1.1 ** b_inc - 1) * 100:.1f} % fewer thefts at equal vehicles, population and area.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for tier in config.GI_TIER_ORDER:
    sub = complete[complete["gi_tier"] == tier]
    if len(sub):
        ns = tier == "not significant"
        ax.scatter(sub["median_income"] / 1000, sub["rate_per_1000"], color=config.GI_TIER_COLORS[tier],
                   s=14 if ns else 26, alpha=0.45 if ns else 0.8, edgecolors="none", zorder=3 if ns else 4,
                   label=f"{tier} (n={len(sub)})")
rho_rate = spearmanr(complete["median_income"], complete["rate_per_1000"]).statistic
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Median household income (2022 $k, log scale)", fontsize=16)
ax.set_ylabel("Thefts per 1,000 household vehicles / yr", fontsize=14)
ax.set_title(f"Theft Rate vs Median Income (Spearman ρ = {rho_rate:.2f}; NB income elasticity {b_inc:.2f}, n = {len(complete)})",
             pad=12, fontsize=13)
ax.set_xticks([20, 30, 50, 70, 100, 150, 200])
ax.xaxis.set_minor_formatter(mticker.NullFormatter())
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.0f}k"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.13), fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig18_income_vs_rate_scatter.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

In [ ]:
geo = tracts_plot[["GEOID", "geometry"]].merge(complete[["GEOID", "residual_log"]], on="GEOID", how="left")
vabs = np.nanpercentile(geo["residual_log"].abs(), 97)
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
geo[geo["residual_log"].isna()].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.2)
geo.dropna(subset=["residual_log"]).plot(ax=ax, column="residual_log", cmap="RdBu_r",
                                          norm=mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs),
                                          edgecolor="#bbb", linewidth=0.15, zorder=3)
config.add_basemap(ax)
ax.set_axis_off()
ax.set_title("Thefts Unexplained by Exposure and Income (NB model 1 residual)", pad=14)
sm_ = cm.ScalarMappable(cmap="RdBu_r", norm=mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs))
plt.colorbar(sm_, ax=ax, fraction=0.03, pad=0.02, shrink=0.75).set_label("log(observed / predicted) thefts")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig19_income_residual_choropleth.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print(complete.nlargest(10, "residual_log")[["GEOID", "median_income", "total_count", "residual_log", "gi_tier"]]
      .round(2).to_string(index=False))